In [1]:
import sys
sys.path.append("../src")
from storage import load_database, DATA_DIR
from queries import count_by_type , history_for_procedure, linked_partner, linked_pairs, last_completed_all

In [2]:
conn = load_database(data_dir=DATA_DIR / 'generated')
for t in ['Frequencies', 'Assets', 'Procedures', 'Service_History']:
    print(t, conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])

Frequencies 4
Assets 40
Procedures 160
Service_History 543


In [3]:
cur = conn.cursor()

count_by_type prototype:

In [4]:
proto_rows = cur.execute("SELECT type, COUNT(*) FROM Assets GROUP BY type ORDER BY COUNT(*) DESC, type").fetchall()
for asset_type, count in proto_rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [5]:
rows = count_by_type(conn)
for asset_type, count in rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [6]:
assert(proto_rows == count_by_type(conn))

history_for_procedure prototype:

In [7]:
proc_id = 'PMP-002-P1'

sql = '''
    SELECT record_id, date_completed, notes FROM Service_History
    WHERE procedure_id = ?
    ORDER BY date_completed
    '''
proto_rows = cur.execute(sql, (proc_id,)).fetchall()
for record_id, date_completed, notes in proto_rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [8]:
rows = history_for_procedure(conn ,'PMP-002-P1')
for record_id, date_completed, notes in rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [9]:
assert(proto_rows == history_for_procedure(conn ,'PMP-002-P1'))

linked_partner prototype:

In [10]:
asset_id = 'PMP-001'

sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id = ?
    ORDER BY a.asset_id
    '''
proto_rows = cur.execute(sql, (asset_id,)).fetchall()
for a_id, a_type, b_id, b_type in proto_rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [11]:
rows = linked_partner(conn, 'PMP-001')
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [12]:
assert(proto_rows == linked_partner(conn ,'PMP-001'))

linked_pairs prototype:

In [13]:
sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id < b.asset_id
    ORDER BY a.asset_id
    '''
proto_rows = cur.execute(sql).fetchall()
for a_id, a_type, b_id, b_type in proto_rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [14]:
rows = linked_pairs(conn)
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [15]:
assert(proto_rows == linked_pairs(conn))

last_completed_all prototype:

In [16]:
sql = '''
    SELECT p.procedure_id, h.last_done
    FROM Procedures as p
    LEFT JOIN  ( 
        SELECT procedure_id, MAX(date_completed)
    AS last_done
        FROM Service_History
        GROUP BY procedure_id
    ) as h
    ON p.procedure_id = h.procedure_id
    ORDER BY p.procedure_id
    '''
proto_rows = cur.execute(sql).fetchall()
print(len(proto_rows))
for last_proc, last_done in proto_rows:
    print(f'{last_proc}, {last_done}')

160
BLR-001-P1, 2026-10-28
BLR-001-P2, 2026-10-21
BLR-001-P3, None
BLR-001-P4, None
BLR-002-P1, 2026-10-29
BLR-002-P2, 2026-09-06
BLR-002-P3, 2026-08-10
BLR-002-P4, None
BLR-003-P1, None
BLR-003-P2, 2026-10-02
BLR-003-P3, 2026-09-04
BLR-003-P4, None
BLR-004-P1, 2026-10-24
BLR-004-P2, None
BLR-004-P3, 2026-10-08
BLR-004-P4, None
GEN-001-P1, None
GEN-001-P2, 2026-08-21
GEN-001-P3, 2026-10-07
GEN-001-P4, None
GEN-002-P1, 2026-10-29
GEN-002-P2, 2026-10-24
GEN-002-P3, None
GEN-002-P4, None
GEN-003-P1, 2026-09-30
GEN-003-P2, 2026-08-25
GEN-003-P3, 2026-10-29
GEN-003-P4, None
GEN-004-P1, 2026-10-29
GEN-004-P2, 2026-10-04
GEN-004-P3, None
GEN-004-P4, None
GEN-005-P1, 2026-10-11
GEN-005-P2, 2026-10-04
GEN-005-P3, 2026-10-09
GEN-005-P4, None
GEN-006-P1, 2026-10-09
GEN-006-P2, 2026-10-21
GEN-006-P3, 2026-09-04
GEN-006-P4, None
GEN-007-P1, 2026-10-29
GEN-007-P2, 2026-10-13
GEN-007-P3, 2026-08-28
GEN-007-P4, None
GEN-008-P1, 2026-10-28
GEN-008-P2, 2026-10-28
GEN-008-P3, 2026-08-16
GEN-008-P4, None


In [17]:
rows = last_completed_all(conn)
print(len(rows))
for last_proc, last_done in rows:
    print(f'{last_proc}, {last_done}')

160
BLR-001-P1, 2026-10-28
BLR-001-P2, 2026-10-21
BLR-001-P3, None
BLR-001-P4, None
BLR-002-P1, 2026-10-29
BLR-002-P2, 2026-09-06
BLR-002-P3, 2026-08-10
BLR-002-P4, None
BLR-003-P1, None
BLR-003-P2, 2026-10-02
BLR-003-P3, 2026-09-04
BLR-003-P4, None
BLR-004-P1, 2026-10-24
BLR-004-P2, None
BLR-004-P3, 2026-10-08
BLR-004-P4, None
GEN-001-P1, None
GEN-001-P2, 2026-08-21
GEN-001-P3, 2026-10-07
GEN-001-P4, None
GEN-002-P1, 2026-10-29
GEN-002-P2, 2026-10-24
GEN-002-P3, None
GEN-002-P4, None
GEN-003-P1, 2026-09-30
GEN-003-P2, 2026-08-25
GEN-003-P3, 2026-10-29
GEN-003-P4, None
GEN-004-P1, 2026-10-29
GEN-004-P2, 2026-10-04
GEN-004-P3, None
GEN-004-P4, None
GEN-005-P1, 2026-10-11
GEN-005-P2, 2026-10-04
GEN-005-P3, 2026-10-09
GEN-005-P4, None
GEN-006-P1, 2026-10-09
GEN-006-P2, 2026-10-21
GEN-006-P3, 2026-09-04
GEN-006-P4, None
GEN-007-P1, 2026-10-29
GEN-007-P2, 2026-10-13
GEN-007-P3, 2026-08-28
GEN-007-P4, None
GEN-008-P1, 2026-10-28
GEN-008-P2, 2026-10-28
GEN-008-P3, 2026-08-16
GEN-008-P4, None


In [18]:
assert(proto_rows == last_completed_all(conn))

In [19]:
conn.close()